In [1]:
!nvidia-smi
!rm -rf /kaggle/working/repo
!git clone https://github.com/rubinanoor/small-model-alignment-starter.git /kaggle/working/repo
%cd /kaggle/working/repo
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest peft trl
!python -m pip install -q -e . --no-deps
import torch, transformers, datasets
print(torch.__version__, transformers.__version__, datasets.__version__, torch.cuda.device_count())

Thu Oct 15 20:52:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# confirm the fix arrived from GitHub
!grep -c "n_train" /kaggle/working/repo/scripts/train_dpo.py

2


In [3]:
# tests
!cd /kaggle/working/repo && PYTHONPATH=. python -m pytest -q tests

............                                                             [100%]
12 passed in 6.01s


In [4]:
import os, shutil, glob
hits = [h for h in glob.glob("/kaggle/input/**/sft/best", recursive=True) if os.path.exists(f"{h}/model.safetensors")]
print(hits)
SRC_BEST = hits[0]
os.makedirs("/kaggle/working/repo/outputs/sft", exist_ok=True)
shutil.copytree(SRC_BEST, "/kaggle/working/repo/outputs/sft/best", dirs_exist_ok=True)
!ls /kaggle/working/repo/outputs/sft/best

['/kaggle/input/notebooks/rubinanoor/assignment-2-notebook-1/repo/outputs/sft/best']
added_tokens.json	merges.txt		 tokenizer.json
chat_template.jinja	model.safetensors	 vocab.json
config.json		special_tokens_map.json
generation_config.json	tokenizer_config.json


In [5]:
# GPU smoke test, about 5 minutes
!cd /kaggle/working/repo && PYTHONPATH=. python scripts/train_dpo.py --dataset hh --out outputs/dpo_smoke --n_train 128 --n_val 16 --accum 4 --eval_every 2

The tokenizer you are loading from 'outputs/sft/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
README.md: 5.77kB [00:00, 18.7MB/s]
harmless-base/train.jsonl.gz: 100%|████████| 13.2M/13.2M [00:00<00:00, 25.9MB/s]
helpful-base/train.jsonl.gz: 100%|█████████| 16.2M/16.2M [00:00<00:00, 59.1MB/s]
helpful-online/train.jsonl.gz: 100%|███████| 20.1M/20.1M [00:00<00:00, 66.0MB/s]
helpful-rejection-sampled/train.jsonl.gz: 100%|█| 25.7M/25.7M [00:00<00:00, 45.5
harmless-base/test.jsonl.gz: 100%|███████████| 743k/743k [00:00<00:00, 4.10MB/s]
helpful-base/test.jsonl.gz: 100%|████████████| 875k/875k [00:00<00:00, 5.45MB/s]
helpful-online/test.jsonl.gz: 100%|████████| 1.05M/1.05M [00:00<00:00, 8.05MB/s]
helpful-reje

In [6]:
# check memory and clean up the smoke test
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv
!rm -rf /kaggle/working/repo/outputs/dpo_smoke

index, memory.used [MiB], memory.total [MiB]
0, 0 MiB, 15360 MiB
1, 0 MiB, 15360 MiB


In [7]:
# full training,
!cd /kaggle/working/repo && PYTHONPATH=. python scripts/train_dpo.py --dataset hh --out outputs/dpo_hh

The tokenizer you are loading from 'outputs/sft/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
train pairs=15000 val pairs=1000 steps/epoch=468
step 1/468 loss=0.6931 acc=0.000 margin=0.0000 gn=40.35 | VAL loss=0.6927 acc=0.507 margin=0.0013 [2.0 min, peak 9.2 GB]
step 2/468 loss=0.6929 acc=0.469 margin=0.0008 gn=45.68 [2.3 min, peak 9.2 GB]
step 3/468 loss=0.6763 acc=0.719 margin=0.0358 gn=42.61 [2.5 min, peak 9.2 GB]
step 4/468 loss=0.6926 acc=0.438 margin=0.0029 gn=35.21 [2.8 min, peak 9.2 GB]
step 5/468 loss=0.6879 acc=0.531 margin=0.0181 gn=41.26 [3.1 min, peak 9.2 GB]
step 6/468 loss=0.6710 acc=0.625 margin=0.0597 gn=39.83 [3.3 min, peak 9.2 GB]
step 7/468 loss=0.7282 acc=0.500 margin=-0.0559 gn

In [8]:
# check the results
!tail -n 3 /kaggle/working/repo/outputs/dpo_hh/metrics.jsonl
!ls /kaggle/working/repo/outputs/dpo_hh /kaggle/working/repo/outputs/dpo_hh/best

{"step": 465, "train_loss": 0.7041991758160293, "train_acc": 0.5, "train_margin": 0.15848994301632047, "grad_norm": 41.34711456298828}
{"step": 466, "train_loss": 0.6159206265583634, "train_acc": 0.625, "train_margin": 0.2903368445113301, "grad_norm": 39.025577545166016}
{"step": 467, "train_loss": 0.6911904816515744, "train_acc": 0.5, "train_margin": 0.13978574448265135, "grad_norm": 46.98637771606445, "val_loss": 0.6500147297680378, "val_acc": 0.626, "val_margin": 0.21512838163226844, "saved_best": true}
/kaggle/working/repo/outputs/dpo_hh:
best  final  metrics.jsonl

/kaggle/working/repo/outputs/dpo_hh/best:
added_tokens.json	merges.txt		 tokenizer.json
chat_template.jinja	model.safetensors	 vocab.json
config.json		special_tokens_map.json
generation_config.json	tokenizer_config.json


In [9]:
# remove the large copied SFT checkpoint, so the saved output stays small
!rm -rf /kaggle/working/repo/outputs/sft
!du -sh /kaggle/working/repo/outputs

3.8G	/kaggle/working/repo/outputs
